# Lab 1 — Reconstruct a Synthetic Patient’s Data Journey
**AI in Healthcare · B.Tech AI, third year · 120 minutes**

Companion: `session-06-data-ecosystem.md`.

You will inspect healthcare records, define row meanings, audit quality, link tables, construct a cohort, parse FHIR JSON, and visualize time. Prerequisites: basic Python and tables; pandas examples are provided.

All records are invented. They are teaching examples, not clinical evidence or a representative population. This lab trains no model and requires no downloads, credentials, or real patient data.

**How to work:** Run cells in order. Complete each **Student response** and exercise cell. Reference answers are at the end. Submit this notebook with outputs and your interpretations. The generated files go to `healthcare_ecosystem_lab_data/`; rerunning the generator recreates only its named teaching files.

**Schedule:** setup/inventory 15 min; quality/dictionary 20; linkage 25; cohort 20; FHIR 20; plots 15; reflection 5.


In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA = Path("healthcare_ecosystem_lab_data")
DATA.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 20)
print("Teaching files:", DATA.resolve())

Teaching files: /home/bk_anupam/code/ML/Bennett/ai_in_healthcare/module1/labs/healthcare_ecosystem_lab_data


If imports fail, install `pandas` and `matplotlib` in your notebook environment (for example, `%pip install pandas matplotlib` in a separate cell), then rerun. No package installation is performed automatically.

In [2]:
patients = pd.DataFrame([
    ("P001", 1980), ("P002", 1992), ("P003", 1975), ("P004", 2000)
], columns=["patient_id", "birth_year"])
encounters = pd.DataFrame([
    ("E001", "P001", "2026-01-10T09:00:00+05:30", "outpatient"),
    ("E002", "P001", "2026-02-10T10:00:00+05:30", "outpatient"),
    ("E003", "P002", "2026-01-15T11:00:00+05:30", "outpatient"),
    ("E004", "P003", "2026-01-20T09:00:00+05:30", "outpatient"),
    ("E005", "P003", "2026-03-01T09:00:00+05:30", "outpatient"),
    ("E006", "P004", "2025-12-20T09:00:00+05:30", "outpatient")
], columns=["encounter_id", "patient_id", "start", "type"])
observations = pd.DataFrame([
    ("O001", "E001", "heart_rate", 78, "beats/min", "2026-01-10T09:10:00+05:30"),
    ("O001", "E001", "heart_rate", 78, "beats/min", "2026-01-10T09:10:00+05:30"),
    ("O002", "E001", "temperature", 37, "Cel", "2026-01-10T09:12:00+05:30"),
    ("O003", "E002", "heart_rate", None, "beats/min", "2026-02-10T10:10:00+05:30"),
    ("O004", "E003", "temperature", 98.6, "[degF]", "2026-01-15T11:10:00+05:30"),
    ("O005", "E004", "heart_rate", 84, "beats/min", "2026-01-20T09:10:00+05:30"),
    ("O006", "E005", "heart_rate", 80, "beats/min", "2026-03-01T09:10:00+05:30")
], columns=["observation_id", "encounter_id", "code", "value", "unit", "effective_time"])
notes = pd.DataFrame([
    ("N001", "P001", "E001", "No history of diabetes. Patient reports reduced activity."),
    ("N002", "P001", "E002", "Follow-up visit. Further assessment planned."),
    ("N003", "P002", "E003", "Family history of diabetes; patient history not established.")
], columns=["note_id", "patient_id", "encounter_id", "text"])
times = pd.date_range("2026-01-10T00:00:00+05:30", periods=24, freq="h")
wearable = pd.DataFrame({"patient_id": "P001", "time": times,
                         "steps": [0,0,0,0,0,20,100,250,300,150,80,120,
                                   200,180,90,140,220,400,350,200,100,40,10,0]})
# Deliberately omit six expected hourly samples. A gap is not a zero.
wearable = wearable.drop(index=range(12,18)).reset_index(drop=True)
for name, frame in {"patients":patients, "encounters":encounters,
                    "observations":observations, "notes":notes, "wearable":wearable}.items():
    frame.to_csv(DATA / f"{name}.csv", index=False)

# Small R4-style collection Bundle, separate from the intentionally messy CSV.
# Educational examples; no formal FHIR/profile validator is run here.
resources = [{"resourceType":"Patient", "id":p} for p in patients.patient_id]
for row in encounters.itertuples():
    resources.append({"resourceType":"Encounter", "id":row.encounter_id,
        "status":"finished", "class":{"system":"http://terminology.hl7.org/CodeSystem/v3-ActCode",
        "code":"AMB"}, "subject":{"reference":f"Patient/{row.patient_id}"},
        "period":{"start":row.start}})
for oid, pid, eid, value, time in [
    ("O001","P001","E001",78,"2026-01-10T09:10:00+05:30"),
    ("O005","P003","E004",84,"2026-01-20T09:10:00+05:30")]:
    resources.append({"resourceType":"Observation", "id":oid, "status":"final",
        "code":{"coding":[{"system":"http://loinc.org", "code":"8867-4", "display":"Heart rate"}]},
        "subject":{"reference":f"Patient/{pid}"},
        "encounter":{"reference":f"Encounter/{eid}"}, "effectiveDateTime":time,
        "valueQuantity":{"value":value, "unit":"beats/minute",
        "system":"http://unitsofmeasure.org", "code":"/min"}})
bundle = {"resourceType":"Bundle", "type":"collection", "entry":[
    {"fullUrl":f"https://example.org/fhir/{r['resourceType']}/{r['id']}", "resource":r}
    for r in resources]}
(DATA / "patient_bundle.json").write_text(json.dumps(bundle, indent=2), encoding="utf-8")
print("Created five CSV files and one JSON file.")

Created five CSV files and one JSON file.


## 1. Inventory (0–15 minutes)
Load the generated files as if another team had supplied them. Identify the meaning of one row in each table. Are these sources, representations, or both?


In [ ]:
tables = {name: pd.read_csv(DATA / f"{name}.csv")
          for name in ["patients", "encounters", "observations", "notes", "wearable"]}

for name, frame in tables.items():
    print(name, frame.shape)
    display(frame.head(3))
    
patients, encounters, observations, notes, wearable = [tables[n] for n in tables]
encounters["start"] = pd.to_datetime(encounters["start"], utc=True)
observations["effective_time"] = pd.to_datetime(observations["effective_time"], utc=True)
wearable["time"] = pd.to_datetime(wearable["time"], utc=True)

patients (4, 2)


,patient_id,birth_year
0,P001,1980
1,P002,1992
2,P003,1975


encounters (6, 4)


,encounter_id,patient_id,start,type
0,E001,P001,2026-01-10T09:00:00+05:30,outpatient
1,E002,P001,2026-02-10T10:00:00+05:30,outpatient
2,E003,P002,2026-01-15T11:00:00+05:30,outpatient


observations (7, 6)


,observation_id,encounter_id,code,value,unit,effective_time
0,O001,E001,heart_rate,78.0,beats/min,2026-01-10T09:10:00+05:30
1,O001,E001,heart_rate,78.0,beats/min,2026-01-10T09:10:00+05:30
2,O002,E001,temperature,37.0,Cel,2026-01-10T09:12:00+05:30


notes (3, 4)


,note_id,patient_id,encounter_id,text
0,N001,P001,E001,No history of diabetes. Patient reports reduce...
1,N002,P001,E002,Follow-up visit. Further assessment planned.
2,N003,P002,E003,Family history of diabetes; patient history no...


wearable (18, 3)


,patient_id,time,steps
0,P001,2026-01-10 00:00:00+05:30,0
1,P001,2026-01-10 01:00:00+05:30,0
2,P001,2026-01-10 02:00:00+05:30,0


**Student response:** For each file, state its row meaning, identifier(s), representation, and a potential use. Explain why the notes CSV still contains unstructured text. Which lecture sources are not represented here?

_Write your answer here._

## 2. Data dictionary and quality audit (15–35 minutes)
Create a dictionary for at least eight columns across three files: meaning, type, unit (if applicable), key role, and missingness interpretation.

Inspect exact duplicate rows, repeated observation IDs, missing values, and measurement units. Do not fill missing measurements with zero.


In [4]:
print("Exact duplicate observation rows:", observations.duplicated().sum())
print("Repeated observation IDs:", observations.observation_id.duplicated().sum())
display(observations.isna().sum().rename("missing_count").to_frame())
display(observations.groupby(["code", "unit"]).size().rename("rows").reset_index())
display(notes.assign(keyword_diabetes=notes.text.str.contains("diabetes", case=False)))

Exact duplicate observation rows: 1
Repeated observation IDs: 1


,missing_count
observation_id,0
encounter_id,0
code,0
value,1
unit,0
effective_time,0


,code,unit,rows
0,heart_rate,beats/min,5
1,temperature,Cel,1
2,temperature,[degF],1


,note_id,patient_id,encounter_id,text,keyword_diabetes
0,N001,P001,E001,No history of diabetes. Patient reports reduce...,True
1,N002,P001,E002,Follow-up visit. Further assessment planned.,False
2,N003,P002,E003,Family history of diabetes; patient history no...,True


In [ ]:
# Exercise: create a dictionary. Add at least eight rows.
data_dictionary = pd.DataFrame(columns=["file", "column", "meaning", "type", "unit", "key_role", "missing_meaning"])
display(data_dictionary)
# Exercise: select the temperature rows and add value_celsius in a COPY.
# Formula for Fahrenheit: (value - 32) * 5 / 9.
# Preserve value and unit so the transformation can be audited.


**Student response:** Identify four quality/interpretation issues. Why is keyword matching unreliable for these notes? What evidence would justify removing a repeated observation ID?

_Write your answer here._

The generator intentionally duplicates one identical export row. We remove exact duplicates for the guided analysis. In real data, a repeated ID with conflicting values requires investigation; a repeated value alone does not justify deletion.


In [ ]:
clean_obs = observations.drop_duplicates().copy()
assert clean_obs.observation_id.is_unique
assert patients.patient_id.is_unique
assert encounters.encounter_id.is_unique
print("Raw rows:", len(observations), "After exact deduplication:", len(clean_obs))

## 3. Link records without changing their meaning (35–60 minutes)
First attach each encounter to a patient, then attach each observation to its encounter. `validate="many_to_one"` checks that the right-hand keys are unique. The merge indicator checks whether references matched; cardinality validation alone does not check that.


In [ ]:
encounter_patient = encounters.merge(patients, on="patient_id", how="left",
                                      validate="many_to_one", indicator=True)
assert encounter_patient["_merge"].eq("both").all()
encounter_patient = encounter_patient.drop(columns="_merge")
linked = clean_obs.merge(encounter_patient, on="encounter_id", how="left",
                         validate="many_to_one", indicator=True)
assert linked["_merge"].eq("both").all()
assert len(linked) == len(clean_obs)
linked = linked.drop(columns="_merge")
display(linked)
print("Rows:", len(linked), "Distinct patients:", linked.patient_id.nunique())

In [ ]:
# Demonstrate multiplication by linking independent encounter and note rows only by patient.
bad_join = encounters.merge(notes[["patient_id", "note_id"]], on="patient_id", how="left")
display(bad_join.loc[bad_join.patient_id.eq("P001"), ["patient_id", "encounter_id", "note_id"]])
# Exercise: join notes to encounters using encounter_id, with appropriate validation.
# Compare P001's row count with the demonstration above.


**Student response:** What does one row of `linked` represent? Why are four distinct patients in the patient table but fewer in `linked`? How would you retain patients without observations? Explain why the demonstration join misassigns notes.

_Write your answer here._

## 4. Define a reproducible cohort (60–80 minutes)
**Rule:** Include patients with at least two distinct encounters starting in the half-open interval **[1 January 2026, 1 March 2026)**, measured in Asia/Kolkata local time. Count encounters, not observations. The end boundary is excluded.

Implement the rule in the next cell. Then report included IDs and explain boundary handling. This is a record-availability cohort, not a disease cohort.


In [ ]:
window_start = pd.Timestamp("2026-01-01", tz="Asia/Kolkata").tz_convert("UTC")
window_end = pd.Timestamp("2026-03-01", tz="Asia/Kolkata").tz_convert("UTC")
# TODO: filter encounters by start, count distinct encounter_id per patient,
# then retain counts >= 2. Name the resulting DataFrame cohort.


**Student response:** State your cohort result and why P003 is excluded. Would counting rows after an observations join give a reliable visit count? Why does this selection not prove anything about disease risk?

_Write your answer here._

## 5. Read FHIR JSON (80–100 minutes)
The collection Bundle packages Patient, Encounter, and Observation resources. The extraction below handles only the simple reference and quantity pattern in this fixture. Production FHIR supports additional reference forms, value types, optional fields, and profiles.

FHIR R4 references: [Observation](https://hl7.org/fhir/R4/observation.html), [Bundle](https://hl7.org/fhir/R4/bundle.html). Successful JSON parsing is not formal FHIR validation.


In [ ]:
bundle_loaded = json.loads((DATA / "patient_bundle.json").read_text(encoding="utf-8"))
resource_index = {f"{e['resource']['resourceType']}/{e['resource']['id']}": e["resource"]
                  for e in bundle_loaded["entry"]}
fhir_rows = []
for resource in resource_index.values():
    if resource["resourceType"] != "Observation":
        continue
    patient_ref = resource["subject"]["reference"]
    encounter_ref = resource["encounter"]["reference"]
    assert patient_ref in resource_index and encounter_ref in resource_index
    assert resource_index[encounter_ref]["subject"]["reference"] == patient_ref
    coding = next(c for c in resource["code"]["coding"] if c["system"] == "http://loinc.org")
    q = resource["valueQuantity"]
    fhir_rows.append({"observation_id":resource["id"],
        "patient_id":patient_ref.split("/")[-1], "encounter_id":encounter_ref.split("/")[-1],
        "loinc_code":coding["code"], "value":q["value"], "ucum_code":q["code"],
        "effective_time":resource["effectiveDateTime"]})
fhir_frame = pd.DataFrame(fhir_rows)
fhir_frame["effective_time"] = pd.to_datetime(fhir_frame["effective_time"], utc=True)
display(fhir_frame)

In [ ]:
# Explicit mappings apply only to the codes/units in this teaching fixture.
comparison_source = linked.copy()
comparison_source["loinc_code"] = comparison_source.code.map({"heart_rate":"8867-4"})
comparison_source["ucum_code"] = comparison_source.unit.map({"beats/min":"/min"})
comparison = fhir_frame.merge(comparison_source, on="observation_id", how="left",
                              suffixes=("_fhir", "_csv"), validate="one_to_one", indicator=True)
assert comparison["_merge"].eq("both").all()
for field in ["patient_id", "encounter_id", "loinc_code", "value", "ucum_code", "effective_time"]:
    assert comparison[f"{field}_fhir"].eq(comparison[f"{field}_csv"]).all(), field
print("All two FHIR observations agree with their CSV counterparts on identity, code, value, unit, and time.")

**Student response:** Locate the subject reference, encounter reference, coding system, and quantity in the JSON file. Why did we compare coded units instead of display labels? Why are there fewer FHIR observations than CSV observations? Does this comparison establish completeness or clinical validity?

_Write your answer here._

## 6. Visualize a journey and missing samples (100–115 minutes)
We know from the synthetic generator that one sample per hour was expected. In real data, verify the expected sampling schedule before calling absent timestamps “missing.” Times below are shown in Asia/Kolkata.


In [ ]:
p1_encounters = encounters.loc[encounters.patient_id.eq("P001")]
p1_obs = linked.loc[linked.patient_id.eq("P001")]
fig, ax = plt.subplots(figsize=(10, 3))
ax.scatter(p1_encounters.start.dt.tz_convert("Asia/Kolkata"), [1]*len(p1_encounters), label="Encounter", marker="s")
ax.scatter(p1_obs.effective_time.dt.tz_convert("Asia/Kolkata"), [0]*len(p1_obs), label="Observation record", marker="x")
ax.set_yticks([0, 1], ["Observation", "Encounter"])
ax.set_title("P001: recorded events (an observation record may have a missing value)")
ax.legend(loc="upper center")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

expected = pd.date_range("2026-01-10", periods=24, freq="h", tz="Asia/Kolkata")
series = wearable.set_index("time")["steps"].tz_convert("Asia/Kolkata").reindex(expected)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(series.index.hour, series.values, marker="o")
ax.axvspan(11.5, 17.5, alpha=0.15, color="orange", label="No samples supplied")
ax.set(xlabel="Hour (Asia/Kolkata)", ylabel="Steps in hourly interval", title="Synthetic wearable activity; gaps remain missing")
ax.legend()
plt.tight_layout()
plt.show()
print("Missing expected hourly samples:", series.isna().sum())

**Student response:** Explain the orange region. What false conclusion might result from replacing these missing samples with zero? Why does the timeline include O003 even though its numeric value is missing?

_Write your answer here._

## 7. Final reflection and submission (115–120 minutes)
Answer in 150–250 words:

1. What information was lost when the FHIR observations were flattened into a table?
2. Which three limitations would matter before training a model on a dataset like this?
3. If predicting at encounter start, would measurements recorded ten minutes later be available? What additional timestamp might be needed in a real system?

**Submit:** executed notebook, completed dictionary, cohort code/result, join exercise, and written responses. Rubric: inventory/dictionary 2; quality interpretation 2; linkage 2; cohort 1; FHIR comparison 1; plots/reflection 2 (10 total).

Synthetic relationships are intentionally simple. Do not use this fixture to evaluate model performance, disease prevalence, or clinical usefulness.

---
## Optional reference answers — consult after attempting the exercises
These cells are executable worked solutions. Your own responses above remain the assessed work.


In [ ]:
temperature_answer = clean_obs.loc[clean_obs.code.eq("temperature")].copy()
assert temperature_answer.unit.isin(["Cel", "[degF]"]).all()
temperature_answer["value_celsius"] = temperature_answer.apply(
    lambda r: (r.value - 32) * 5 / 9 if r.unit == "[degF]" else r.value, axis=1)
display(temperature_answer)

correct_notes_answer = encounters.merge(notes[["encounter_id", "note_id", "text"]],
                                        on="encounter_id", how="left", validate="one_to_many")
assert len(correct_notes_answer.loc[correct_notes_answer.patient_id.eq("P001")]) == 2

eligible_answer = encounters.loc[encounters.start.ge(window_start) & encounters.start.lt(window_end)]
counts_answer = eligible_answer.groupby("patient_id").encounter_id.nunique()
cohort_answer = counts_answer[counts_answer.ge(2)].rename("encounter_count").reset_index()
display(cohort_answer)
assert cohort_answer.patient_id.tolist() == ["P001"]
assert len(clean_obs) == 6
assert linked.patient_id.nunique() == 3
assert series.isna().sum() == 6
print("Reference checks passed.")

### Interpretation guide

- Quality issues include one exact duplicate, one missing value, temperature values using different units, and note wording that defeats simple disease keyword flags. The wearable adds a six-sample gap.
- `linked` has one row per deduplicated observation. P004 has an encounter but no observation rows. Start from patients and use left joins if retaining all patients is the goal; document the resulting granularity.
- The patient-only join pairs each of P001's two visits with both notes, producing four rows. Encounter linkage preserves each note's visit. Multiple legitimate notes per encounter could still expand rows.
- P001 has two visits in the cohort interval. P003's second visit is on the excluded March 1 boundary. Count distinct encounters to avoid counting multiple observations as visits.
- FHIR is deliberately a subset with two observations. Agreement on selected fields does not establish completeness. Flattening drops status, full URLs, coding-system details, and other resource context from the original JSON.
- A gap in supplied wearable samples is not evidence of inactivity. O003 is a documented observation row whose value is absent; it still has an event time.
- Event time and availability time can differ. Inputs must have been available at the intended prediction time.
- A useful dictionary describes `patient_id` as a synthetic person key, `encounter_id` as a visit key, `observation_id` as an observation key, `effective_time` as measurement time, and `value` together with `code` and `unit`. Missing meanings must not be invented when the source does not specify them.
